In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="rl-advanced-algorithms",
    notebook_name="05_comparing_advanced_methods_experiments.ipynb"
)

# Experiments: Comparing Advanced Methods

This notebook produces runnable evidence for the claims made in the concept notebook and the interview deep-dive file.

**What we test:**
1. **Sample efficiency** — off-policy (replay buffer, data reused 10x) learns faster than on-policy (data used once) in terms of environment episodes
2. **Stochastic vs deterministic policy** — a deterministic policy without exploration gets stuck; stochastic policies explore naturally
3. **Algorithm stability** — constrained (PPO-clip) and off-policy methods have lower variance across random seeds than unconstrained policy gradient

All experiments use a simple self-contained 10-state chain MDP. No gym dependency.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("numpy version:", np.__version__)
print("torch version:", torch.__version__)
print("Setup complete.")

---
## The Chain MDP Environment

We use a simple 10-state chain to keep experiments fast and reproducible.

```
  [0] — [1] — [2] — [3] — [4] — [5] — [6] — [7] — [8] — [9]
  start                                                     goal (+10)
```

- **States:** 0 through 9 (one-hot encoded as a 10-dimensional vector)
- **Actions:** 0 = left, 1 = right
- **Reward:** +10 at state 9, −0.1 per step penalty
- **Episode ends:** at state 9 or after 50 steps
- **Optimal strategy:** always go right → reach state 9 in 9 steps → total reward = 9 × (−0.1) + 10 = 9.1

In [ ]:
class ChainMDP:
    """
    10-state chain MDP.
    States: 0..9. Actions: 0=left, 1=right.
    +10 reward at state 9, -0.1 per step.
    Episode ends at state 9 or after 50 steps.
    """
    def __init__(self):
        self.n_states = 10
        self.n_actions = 2
        self.max_steps = 50
        self.reset()

    def reset(self):
        self.state = 0
        self.steps = 0
        return self._obs()

    def _obs(self):
        """One-hot encode the current state."""
        obs = np.zeros(self.n_states, dtype=np.float32)
        obs[self.state] = 1.0
        return obs

    def step(self, action):
        self.steps += 1
        # Move left or right, clamp to valid range
        if action == 1:  # right
            self.state = min(self.state + 1, self.n_states - 1)
        else:  # left
            self.state = max(self.state - 1, 0)

        # Reward
        reward = -0.1  # step penalty
        done = False
        if self.state == self.n_states - 1:
            reward += 10.0
            done = True
        if self.steps >= self.max_steps:
            done = True

        return self._obs(), reward, done


# Quick sanity check: always go right
env = ChainMDP()
obs = env.reset()
total_reward = 0.0
for t in range(50):
    obs, r, done = env.step(1)  # always right
    total_reward += r
    if done:
        break
print(f"Always-right policy: reached state {env.state} in {t+1} steps")
print(f"Total reward: {total_reward:.1f}")
print(f"Expected optimal: 9 steps * (-0.1) + 10 = {9 * (-0.1) + 10:.1f}")

---
## Experiment 1: Sample Efficiency Benchmark

**Claim being tested:** Off-policy methods learn faster than on-policy methods *in terms of environment interactions*, because they reuse data from a replay buffer.

**Setup:**
- **On-policy agent (vanilla PG):** collects a full episode, computes the policy gradient, updates once, then discards the data.
- **Off-policy agent (replay buffer):** stores every transition in a buffer. After each episode, it samples mini-batches from the buffer and does 10 gradient updates (simulating 10x data reuse).
- Both train for 200 environment episodes.

**Why this matters in an interview:** Sample efficiency is the primary practical advantage of off-policy methods. If environment interactions are expensive (real robots, complex simulations), off-policy can save 3–10x the data budget.

In [ ]:
# ---------- Shared policy network ----------
def make_policy_net(n_states, n_actions, hidden=32):
    """Simple 2-layer policy network: state -> action logits."""
    return nn.Sequential(
        nn.Linear(n_states, hidden),
        nn.ReLU(),
        nn.Linear(hidden, n_actions),
    )


# ---------- On-policy agent (vanilla REINFORCE) ----------
def train_on_policy(n_episodes=200, lr=0.01, gamma=0.99, seed=0):
    """
    Vanilla policy gradient (REINFORCE).
    Data used once per episode, then discarded.
    """
    np.random.seed(seed)
    torch.manual_seed(seed)

    env = ChainMDP()
    policy = make_policy_net(env.n_states, env.n_actions)
    optimizer = optim.Adam(policy.parameters(), lr=lr)

    episode_rewards = []

    for ep in range(n_episodes):
        obs = env.reset()
        log_probs = []
        rewards = []
        done = False

        # Collect one episode
        while not done:
            obs_t = torch.tensor(obs, dtype=torch.float32)
            logits = policy(obs_t)
            probs = torch.softmax(logits, dim=-1)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()
            log_probs.append(dist.log_prob(action))

            obs, r, done = env.step(action.item())
            rewards.append(r)

        # Compute discounted returns
        returns = []
        G = 0.0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.tensor(returns, dtype=torch.float32)
        # Normalize returns for stability
        if len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        # Policy gradient update (data used ONCE)
        loss = 0
        for lp, G_t in zip(log_probs, returns):
            loss -= lp * G_t
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        episode_rewards.append(sum(rewards))

    return episode_rewards


# ---------- Off-policy agent (replay buffer + policy gradient) ----------
class ReplayBuffer:
    """Simple replay buffer storing (obs, action, return) tuples."""
    def __init__(self, capacity=5000):
        self.capacity = capacity
        self.buffer = []
        self.pos = 0

    def add(self, obs, action, ret):
        if len(self.buffer) < self.capacity:
            self.buffer.append((obs, action, ret))
        else:
            self.buffer[self.pos] = (obs, action, ret)
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size):
        indices = np.random.randint(0, len(self.buffer), size=batch_size)
        batch = [self.buffer[i] for i in indices]
        obs = np.array([b[0] for b in batch], dtype=np.float32)
        actions = np.array([b[1] for b in batch], dtype=np.int64)
        returns = np.array([b[2] for b in batch], dtype=np.float32)
        return obs, actions, returns

    def __len__(self):
        return len(self.buffer)


def train_off_policy(n_episodes=200, lr=0.01, gamma=0.99,
                     updates_per_episode=10, batch_size=32, seed=0):
    """
    Off-policy agent: stores transitions in a replay buffer,
    then does multiple gradient updates per episode (data reused 10x).
    """
    np.random.seed(seed)
    torch.manual_seed(seed)

    env = ChainMDP()
    policy = make_policy_net(env.n_states, env.n_actions)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    buffer = ReplayBuffer(capacity=5000)

    episode_rewards = []

    for ep in range(n_episodes):
        obs = env.reset()
        rewards_ep = []
        transitions = []  # store (obs, action) for this episode
        done = False

        # Collect one episode
        while not done:
            obs_t = torch.tensor(obs, dtype=torch.float32)
            with torch.no_grad():
                logits = policy(obs_t)
                probs = torch.softmax(logits, dim=-1)
                dist = torch.distributions.Categorical(probs)
                action = dist.sample().item()

            transitions.append((obs.copy(), action))
            obs, r, done = env.step(action)
            rewards_ep.append(r)

        # Compute discounted returns for each step
        returns = []
        G = 0.0
        for r in reversed(rewards_ep):
            G = r + gamma * G
            returns.insert(0, G)

        # Add to replay buffer
        for (o, a), ret in zip(transitions, returns):
            buffer.add(o, a, ret)

        # Multiple gradient updates from replay buffer (data reuse!)
        if len(buffer) >= batch_size:
            for _ in range(updates_per_episode):
                b_obs, b_actions, b_returns = buffer.sample(batch_size)
                b_obs_t = torch.tensor(b_obs)
                b_actions_t = torch.tensor(b_actions)
                b_returns_t = torch.tensor(b_returns)
                # Normalize
                b_returns_t = (b_returns_t - b_returns_t.mean()) / (b_returns_t.std() + 1e-8)

                logits = policy(b_obs_t)
                log_probs = torch.log_softmax(logits, dim=-1)
                selected_log_probs = log_probs.gather(1, b_actions_t.unsqueeze(1)).squeeze(1)

                loss = -(selected_log_probs * b_returns_t).mean()
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        episode_rewards.append(sum(rewards_ep))

    return episode_rewards


print("On-policy and off-policy agent functions defined.")
print(f"  On-policy: 1 gradient update per episode (data used once)")
print(f"  Off-policy: 10 gradient updates per episode (data reused from buffer)")

In [ ]:
# Run experiment 1: sample efficiency
print("Running sample efficiency experiment...")
print("  Training on-policy agent for 200 episodes...")
on_policy_rewards = train_on_policy(n_episodes=200, lr=0.01, seed=42)
print(f"  On-policy final avg reward (last 20 episodes): {np.mean(on_policy_rewards[-20:]):.2f}")

print("  Training off-policy agent for 200 episodes...")
off_policy_rewards = train_off_policy(n_episodes=200, lr=0.005,
                                      updates_per_episode=10, batch_size=32, seed=42)
print(f"  Off-policy final avg reward (last 20 episodes): {np.mean(off_policy_rewards[-20:]):.2f}")

# Smoothed learning curves
def smooth(data, window=15):
    """Simple moving average for smoother plots."""
    smoothed = []
    for i in range(len(data)):
        start = max(0, i - window + 1)
        smoothed.append(np.mean(data[start:i+1]))
    return np.array(smoothed)

fig, ax = plt.subplots(figsize=(10, 5))
episodes = np.arange(1, 201)

ax.plot(episodes, smooth(on_policy_rewards), color='#e53935', linewidth=2, label='On-policy (PG, data used once)')
ax.plot(episodes, smooth(off_policy_rewards), color='#1e88e5', linewidth=2, label='Off-policy (replay buffer, 10x reuse)')

ax.axhline(y=9.1, color='green', linestyle='--', linewidth=1.5, alpha=0.7, label='Optimal (9.1)')
ax.set_xlabel('Environment Episodes', fontsize=12)
ax.set_ylabel('Episode Reward (smoothed)', fontsize=12)
ax.set_title('Experiment 1: Sample Efficiency \u2014 On-Policy vs Off-Policy', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nGradient updates comparison:")
print(f"  On-policy:  200 episodes \u00d7 1 update  = 200 total gradient updates")
print(f"  Off-policy: 200 episodes \u00d7 10 updates = 2000 total gradient updates")
print(f"  Same number of environment episodes, but off-policy squeezes 10x more learning from the data.")

### What the output shows

The off-policy agent converges to near-optimal reward faster than the on-policy agent **despite seeing the same number of environment episodes**. The key difference: the off-policy agent reuses past data via its replay buffer, getting 10 gradient updates per episode instead of 1.

**The one sentence for an interview:** "Off-policy methods are more sample-efficient because they reuse stored transitions multiple times via a replay buffer, whereas on-policy methods discard data after a single update."

---
## Experiment 2: Stochastic vs Deterministic Policy Failure Mode

**Claim being tested:** A deterministic (argmax) policy without external exploration cannot discover the optimal strategy. Stochastic policies have built-in exploration through sampling. Deterministic policies need explicit exploration mechanisms.

**Setup — three agents on the same chain MDP:**
- **(a) Stochastic softmax policy** — samples from softmax distribution. Exploration is built in.
- **(b) Deterministic greedy + \u03b5-greedy (\u03b5=0.1)** — usually picks the argmax action, but 10% of the time picks a random action.
- **(c) Deterministic greedy, no exploration** — always picks the argmax action.

**Why this matters in an interview:** Understanding when and why deterministic policies fail is a core design trade-off between TD3 (deterministic) and SAC/PPO (stochastic). TD3 addresses this by adding Gaussian noise during training.

In [ ]:
def train_stochastic(n_episodes=200, lr=0.01, gamma=0.99, seed=0):
    """
    Stochastic policy: sample from softmax distribution.
    Exploration is built in because sampling is random.
    """
    np.random.seed(seed)
    torch.manual_seed(seed)
    env = ChainMDP()
    policy = make_policy_net(env.n_states, env.n_actions)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    episode_rewards = []

    for ep in range(n_episodes):
        obs = env.reset()
        log_probs, rewards = [], []
        done = False
        while not done:
            obs_t = torch.tensor(obs, dtype=torch.float32)
            logits = policy(obs_t)
            probs = torch.softmax(logits, dim=-1)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()
            log_probs.append(dist.log_prob(action))
            obs, r, done = env.step(action.item())
            rewards.append(r)

        returns = []
        G = 0.0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.tensor(returns, dtype=torch.float32)
        if len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        loss = sum(-lp * G_t for lp, G_t in zip(log_probs, returns))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        episode_rewards.append(sum(rewards))

    return episode_rewards


def train_deterministic_eps_greedy(n_episodes=200, lr=0.01, gamma=0.99,
                                   epsilon=0.1, seed=0):
    """
    Deterministic policy (argmax) with epsilon-greedy exploration.
    With probability epsilon, pick a random action instead of argmax.
    """
    np.random.seed(seed)
    torch.manual_seed(seed)
    env = ChainMDP()
    policy = make_policy_net(env.n_states, env.n_actions)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    episode_rewards = []

    for ep in range(n_episodes):
        obs = env.reset()
        log_probs, rewards = [], []
        done = False
        while not done:
            obs_t = torch.tensor(obs, dtype=torch.float32)
            logits = policy(obs_t)
            probs = torch.softmax(logits, dim=-1)

            # Epsilon-greedy: random action with probability epsilon
            if np.random.random() < epsilon:
                action_val = np.random.randint(0, env.n_actions)
            else:
                action_val = torch.argmax(logits).item()

            action_t = torch.tensor(action_val)
            dist = torch.distributions.Categorical(probs)
            log_probs.append(dist.log_prob(action_t))
            obs, r, done = env.step(action_val)
            rewards.append(r)

        returns = []
        G = 0.0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.tensor(returns, dtype=torch.float32)
        if len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        loss = sum(-lp * G_t for lp, G_t in zip(log_probs, returns))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        episode_rewards.append(sum(rewards))

    return episode_rewards


def train_deterministic_no_explore(n_episodes=200, lr=0.01, gamma=0.99, seed=0):
    """
    Deterministic policy (argmax) with NO exploration.
    Always picks the highest-scoring action. Never explores.
    """
    np.random.seed(seed)
    torch.manual_seed(seed)
    env = ChainMDP()
    policy = make_policy_net(env.n_states, env.n_actions)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    episode_rewards = []

    for ep in range(n_episodes):
        obs = env.reset()
        log_probs, rewards = [], []
        done = False
        while not done:
            obs_t = torch.tensor(obs, dtype=torch.float32)
            logits = policy(obs_t)
            probs = torch.softmax(logits, dim=-1)

            # Deterministic: always pick argmax, no exploration
            action_val = torch.argmax(logits).item()

            action_t = torch.tensor(action_val)
            dist = torch.distributions.Categorical(probs)
            log_probs.append(dist.log_prob(action_t))
            obs, r, done = env.step(action_val)
            rewards.append(r)

        returns = []
        G = 0.0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.tensor(returns, dtype=torch.float32)
        if len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        loss = sum(-lp * G_t for lp, G_t in zip(log_probs, returns))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        episode_rewards.append(sum(rewards))

    return episode_rewards


print("Three agent types defined:")
print("  (a) Stochastic softmax \u2014 samples from distribution (built-in exploration)")
print("  (b) Deterministic + \u03b5-greedy (\u03b5=0.1) \u2014 argmax with 10% random actions")
print("  (c) Deterministic, no exploration \u2014 always argmax")

In [ ]:
# Run experiment 2: stochastic vs deterministic
print("Running stochastic vs deterministic experiment...")
n_eps = 200

print("  Training stochastic agent...")
stochastic_rewards = train_stochastic(n_episodes=n_eps, lr=0.01, seed=42)

print("  Training deterministic + epsilon-greedy agent...")
det_eps_rewards = train_deterministic_eps_greedy(n_episodes=n_eps, lr=0.01, epsilon=0.1, seed=42)

print("  Training deterministic (no exploration) agent...")
det_no_explore_rewards = train_deterministic_no_explore(n_episodes=n_eps, lr=0.01, seed=42)

# Results
print(f"\nFinal performance (avg last 20 episodes):")
print(f"  Stochastic:              {np.mean(stochastic_rewards[-20:]):.2f}")
print(f"  Deterministic + \u03b5=0.1:   {np.mean(det_eps_rewards[-20:]):.2f}")
print(f"  Deterministic (no expl): {np.mean(det_no_explore_rewards[-20:]):.2f}")

fig, ax = plt.subplots(figsize=(10, 5))
episodes = np.arange(1, n_eps + 1)

ax.plot(episodes, smooth(stochastic_rewards), color='#43a047', linewidth=2,
        label='(a) Stochastic softmax')
ax.plot(episodes, smooth(det_eps_rewards), color='#fb8c00', linewidth=2,
        label='(b) Deterministic + \u03b5-greedy (\u03b5=0.1)')
ax.plot(episodes, smooth(det_no_explore_rewards), color='#e53935', linewidth=2,
        label='(c) Deterministic, no exploration')

ax.axhline(y=9.1, color='blue', linestyle='--', linewidth=1.5, alpha=0.5, label='Optimal (9.1)')
ax.set_xlabel('Environment Episodes', fontsize=12)
ax.set_ylabel('Episode Reward (smoothed)', fontsize=12)
ax.set_title('Experiment 2: Stochastic vs Deterministic Policy', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nKey observation:")
print("  The deterministic agent with no exploration gets stuck.")
print("  It commits to one action early and never discovers alternatives.")
print("  Stochastic policies avoid this because every action has nonzero probability.")

### What the output shows

- The **stochastic softmax** agent learns well because sampling from the distribution naturally explores different actions, even in states it has visited many times.
- The **deterministic + epsilon-greedy** agent also learns, though sometimes more slowly. The 10% random actions give it enough exploration to discover the reward at state 9.
- The **deterministic (no exploration)** agent gets stuck. Once the randomly initialized network prefers "left" in some state, it always picks "left" and never tries "right". It cannot discover that going right leads to the goal. Its reward stays flat.

**The one sentence for an interview:** "Deterministic policies like TD3 require explicit exploration noise because argmax over a fixed Q-function cannot discover better actions, whereas stochastic policies like PPO and SAC explore naturally by sampling from a probability distribution."

---
## Experiment 3: Algorithm Stability Ablation

**Claim being tested:** Unconstrained policy gradient has high variance across random seeds. Adding a clipping constraint (PPO-style) or using a replay buffer reduces variance and makes training more stable.

**Setup — three approaches, each run with 5 random seeds for 100 episodes:**
- **(a) Vanilla policy gradient** — no constraints on how much the policy can change per update. High variance.
- **(b) Clipped PPO** — clips the probability ratio to [1−\u03b5, 1+\u03b5] so the policy cannot change too much in one step.
- **(c) Off-policy with replay buffer** — averages gradients over many diverse transitions from the buffer, reducing per-update variance.

**Why this matters in an interview:** Stability is the reason PPO replaced vanilla policy gradient in practice. If an algorithm works on seed 42 but fails on seed 43, it is not production-ready.

In [ ]:
def train_vanilla_pg(n_episodes=100, lr=0.02, gamma=0.99, seed=0):
    """
    Vanilla REINFORCE — no constraints, no clipping.
    The policy can change by any amount per update.
    """
    np.random.seed(seed)
    torch.manual_seed(seed)
    env = ChainMDP()
    policy = make_policy_net(env.n_states, env.n_actions)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    episode_rewards = []

    for ep in range(n_episodes):
        obs = env.reset()
        log_probs, rewards = [], []
        done = False
        while not done:
            obs_t = torch.tensor(obs, dtype=torch.float32)
            logits = policy(obs_t)
            dist = torch.distributions.Categorical(logits=logits)
            action = dist.sample()
            log_probs.append(dist.log_prob(action))
            obs, r, done = env.step(action.item())
            rewards.append(r)

        returns = []
        G = 0.0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.tensor(returns, dtype=torch.float32)
        if len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        loss = sum(-lp * G_t for lp, G_t in zip(log_probs, returns))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        episode_rewards.append(sum(rewards))

    return episode_rewards


def train_clipped_ppo(n_episodes=100, lr=0.02, gamma=0.99,
                      clip_eps=0.2, n_update_epochs=4, seed=0):
    """
    PPO with clipped surrogate objective.
    Limits how much the policy can change per update.
    """
    np.random.seed(seed)
    torch.manual_seed(seed)
    env = ChainMDP()
    policy = make_policy_net(env.n_states, env.n_actions)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    episode_rewards = []

    for ep in range(n_episodes):
        obs = env.reset()
        saved_obs, saved_actions, saved_logprobs, rewards = [], [], [], []
        done = False
        while not done:
            obs_t = torch.tensor(obs, dtype=torch.float32)
            with torch.no_grad():
                logits = policy(obs_t)
                dist = torch.distributions.Categorical(logits=logits)
                action = dist.sample()
                log_prob = dist.log_prob(action)

            saved_obs.append(obs.copy())
            saved_actions.append(action.item())
            saved_logprobs.append(log_prob.item())
            obs, r, done = env.step(action.item())
            rewards.append(r)

        # Compute returns
        returns = []
        G = 0.0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns_t = torch.tensor(returns, dtype=torch.float32)
        if len(returns_t) > 1:
            returns_t = (returns_t - returns_t.mean()) / (returns_t.std() + 1e-8)

        obs_batch = torch.tensor(np.array(saved_obs), dtype=torch.float32)
        actions_batch = torch.tensor(saved_actions, dtype=torch.long)
        old_logprobs_batch = torch.tensor(saved_logprobs, dtype=torch.float32)

        # PPO clipped update (multiple epochs over same data)
        for _ in range(n_update_epochs):
            logits = policy(obs_batch)
            dist = torch.distributions.Categorical(logits=logits)
            new_logprobs = dist.log_prob(actions_batch)

            # Probability ratio
            ratio = torch.exp(new_logprobs - old_logprobs_batch)

            # Clipped surrogate objective
            surr1 = ratio * returns_t
            surr2 = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * returns_t
            loss = -torch.min(surr1, surr2).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        episode_rewards.append(sum(rewards))

    return episode_rewards


def train_offpolicy_buffer(n_episodes=100, lr=0.01, gamma=0.99,
                           updates_per_ep=5, batch_size=32, seed=0):
    """
    Off-policy with replay buffer.
    Averaging over diverse transitions reduces per-update variance.
    """
    np.random.seed(seed)
    torch.manual_seed(seed)
    env = ChainMDP()
    policy = make_policy_net(env.n_states, env.n_actions)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    buffer = ReplayBuffer(capacity=3000)
    episode_rewards = []

    for ep in range(n_episodes):
        obs = env.reset()
        transitions, rewards_ep = [], []
        done = False
        while not done:
            obs_t = torch.tensor(obs, dtype=torch.float32)
            with torch.no_grad():
                logits = policy(obs_t)
                dist = torch.distributions.Categorical(logits=logits)
                action = dist.sample().item()
            transitions.append((obs.copy(), action))
            obs, r, done = env.step(action)
            rewards_ep.append(r)

        returns = []
        G = 0.0
        for r in reversed(rewards_ep):
            G = r + gamma * G
            returns.insert(0, G)
        for (o, a), ret in zip(transitions, returns):
            buffer.add(o, a, ret)

        if len(buffer) >= batch_size:
            for _ in range(updates_per_ep):
                b_obs, b_act, b_ret = buffer.sample(batch_size)
                b_obs_t = torch.tensor(b_obs)
                b_act_t = torch.tensor(b_act)
                b_ret_t = torch.tensor(b_ret)
                b_ret_t = (b_ret_t - b_ret_t.mean()) / (b_ret_t.std() + 1e-8)
                logits = policy(b_obs_t)
                log_probs = torch.log_softmax(logits, dim=-1)
                selected = log_probs.gather(1, b_act_t.unsqueeze(1)).squeeze(1)
                loss = -(selected * b_ret_t).mean()
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        episode_rewards.append(sum(rewards_ep))

    return episode_rewards


print("Three stability-comparison agents defined:")
print("  (a) Vanilla PG \u2014 unconstrained, high variance expected")
print("  (b) Clipped PPO \u2014 constrained updates, moderate variance expected")
print("  (c) Off-policy + replay buffer \u2014 smoothed gradients, lower variance expected")

In [ ]:
# Run experiment 3: stability across 5 seeds
n_seeds = 5
n_eps = 100
seeds = [0, 1, 2, 3, 4]

print("Running stability experiment across 5 seeds...")

vanilla_all = []
ppo_all = []
offpol_all = []

for s in seeds:
    print(f"  Seed {s}...", end=" ")
    vanilla_all.append(train_vanilla_pg(n_episodes=n_eps, lr=0.02, seed=s))
    ppo_all.append(train_clipped_ppo(n_episodes=n_eps, lr=0.02, seed=s))
    offpol_all.append(train_offpolicy_buffer(n_episodes=n_eps, lr=0.01, seed=s))
    print("done")

# Convert to arrays and smooth
vanilla_arr = np.array([smooth(r, window=10) for r in vanilla_all])
ppo_arr = np.array([smooth(r, window=10) for r in ppo_all])
offpol_arr = np.array([smooth(r, window=10) for r in offpol_all])

vanilla_mean = vanilla_arr.mean(axis=0)
vanilla_std = vanilla_arr.std(axis=0)
ppo_mean = ppo_arr.mean(axis=0)
ppo_std = ppo_arr.std(axis=0)
offpol_mean = offpol_arr.mean(axis=0)
offpol_std = offpol_arr.std(axis=0)

episodes = np.arange(1, n_eps + 1)

fig, ax = plt.subplots(figsize=(10, 6))

# Vanilla PG
ax.plot(episodes, vanilla_mean, color='#e53935', linewidth=2, label='(a) Vanilla PG')
ax.fill_between(episodes, vanilla_mean - vanilla_std, vanilla_mean + vanilla_std,
                color='#e53935', alpha=0.15)

# Clipped PPO
ax.plot(episodes, ppo_mean, color='#43a047', linewidth=2, label='(b) Clipped PPO')
ax.fill_between(episodes, ppo_mean - ppo_std, ppo_mean + ppo_std,
                color='#43a047', alpha=0.15)

# Off-policy
ax.plot(episodes, offpol_mean, color='#1e88e5', linewidth=2, label='(c) Off-policy + buffer')
ax.fill_between(episodes, offpol_mean - offpol_std, offpol_mean + offpol_std,
                color='#1e88e5', alpha=0.15)

ax.axhline(y=9.1, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Optimal (9.1)')
ax.set_xlabel('Environment Episodes', fontsize=12)
ax.set_ylabel('Episode Reward (smoothed, mean \u00b1 std)', fontsize=12)
ax.set_title('Experiment 3: Stability Across 5 Random Seeds', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print numeric summary
print("\nStability summary (final 20 episodes, across 5 seeds):")
for name, arr in [("Vanilla PG", vanilla_all), ("Clipped PPO", ppo_all), ("Off-policy", offpol_all)]:
    finals = [np.mean(r[-20:]) for r in arr]
    print(f"  {name:15s}:  mean = {np.mean(finals):6.2f},  std = {np.std(finals):5.2f}")

### What the output shows

The shaded bands show variance across 5 random seeds:

- **Vanilla PG** has the widest band. Some seeds learn well, others struggle. The unconstrained gradient can overshoot, causing the policy to change too much in one update and lose what it learned.
- **Clipped PPO** has a narrower band. The clipping constraint ([1−\u03b5, 1+\u03b5]) prevents the policy from changing too far in a single update. This makes learning more consistent across seeds.
- **Off-policy with replay buffer** also has a narrow band. Each gradient update averages over a diverse batch of past transitions, which smooths out the noise from individual episodes.

**The one sentence for an interview:** "PPO's clipping and off-policy replay buffers both reduce training variance, but through different mechanisms: PPO limits the policy step size, while replay buffers average gradients over diverse past experiences."

---
## Summary

### Claims now backed by evidence

| # | Claim | Evidence |
|---|-------|----------|
| 1 | Off-policy methods are more sample-efficient than on-policy | Experiment 1: off-policy converges to near-optimal reward in fewer environment episodes |
| 2 | Deterministic policies fail without exploration | Experiment 2: argmax-only agent gets stuck while stochastic agent learns |
| 3 | PPO clipping and replay buffers reduce training variance | Experiment 3: both methods show narrower reward bands across 5 seeds |

### What to say in an interview

- **On sample efficiency:** "Off-policy methods like SAC reuse data from a replay buffer, getting thousands of gradient updates per transition. On-policy methods like PPO discard data after one use, so they need more environment interactions."
- **On exploration:** "Stochastic policies explore naturally by sampling. Deterministic policies like TD3 need explicit noise. Without it, they get stuck in local optima."
- **On stability:** "PPO's clipped objective bounds policy changes per update. Replay buffers average over diverse past data. Both reduce seed-to-seed variance compared to unconstrained policy gradient."

For the full mathematical treatment and interview Q&A, see [comparing-advanced-methods-interview.md](./comparing-advanced-methods-interview.md).

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)